# Import Geopolitical Risk Index (GPR)
- Author: Bryan Bravo
- Created: 2026-03-23
## Import Libraries

In [1]:
import os
import sys

# Set JAVA_HOME before importing PySpark and use findspark
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-22'  # May need to remove or update in cloud environment.
import findspark
findspark.init()

import requests
import pandas as pd
import numpy as np
import json
import pyspark
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from functools import reduce
from pyspark.sql import (
    functions as F,
    Window as W,
    types as T,
    SparkSession,
    DataFrame
)

# api keys and other hardcoded values for the Strait of Hormuz Research project.
# Note: In a production environment, these should be stored securely and not hardcoded.
import hardcoded_keys # Note: This file is added to .gitignore to prevent accidental commits of sensitive information.

import proj_vars

### Initialize Spark Session

In [2]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BusinessPlanAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.parquet.nativeio.enabled", "false") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


## Query

In [3]:
# Map country with country code
country_mapping = {
    'australia': 'AUS',
    'brazil': 'BRA',
    'canada': 'CAN',
    'china': 'CHN',
    # 'euro': 'EU',  # No IMF Data, must source from elsewhere.
    'france': 'FRA',
    'germany': 'DEU',
    'india': 'IND',
    'italy': 'ITA',
    'japan': 'JPN',
    'mexico': 'MEX',
    'south_korea': 'KOR',
    'russia': 'RUS',
    'south_africa': 'ZAF',
    'turkiye': 'TUR',
    'united_kingdom': 'GBR',
    'united_states': 'USA'
}

In [4]:
# Import `xls` from website
raw_df = pd.read_excel("https://www.matteoiacoviello.com/gpr_files/data_gpr_export.xls",
                       header=0,
                       engine='xlrd')
raw_df.columns = [col.lower() for col in raw_df.columns]

# subset df to include dates after `2006-01-01`
raw_df = raw_df.loc[:, 'month':'gprhc_zaf']
raw_df['month'] = raw_df['month'].dt.strftime('%Y%m%d').astype(int)
raw_df = raw_df[raw_df['month']>=20060101]

In [ ]:
# Convert pandas df to spark df
gpr_df = spark.createDataFrame(raw_df)

# Melt country columns into a single `gpr_index` variable
gpr_df = (
    gpr_df.melt(
        ids=['month'], 
        values=[f"gprc_{country_code.lower()}" for country_code in country_mapping.values()],
        variableColumnName='country', valueColumnName='gpr_index'
    )
)

# remap values in `country` variable.
gpr_df = (
    gpr_df
    .withColumn('country', F.split(F.col('country'), '_')[1])
    .replace({cntry_code.lower(): cntry_name for cntry_name, cntry_code in country_mapping.items()}, subset=['country'])
)

# separate joining variables
gpr_df = (
    gpr_df
    .withColumns({
        'year': F.substring(F.col('month').cast('string'), 1, 4).cast('int'),
        'month': F.substring(F.col('month').cast('string'), 5, 2).cast('int')
    })
    .select('country', 'year', 'month', 'gpr_index')
)

In [6]:
gpr_df.show()

+--------------+----+-----+-------------------+
|       country|year|month|          gpr_index|
+--------------+----+-----+-------------------+
|     australia|2006|    1|0.06155740097165108|
|        brazil|2006|    1| 0.0492459200322628|
|        canada|2006|    1|0.12311480194330215|
|         china|2006|    1| 0.6309633851051331|
|        france|2006|    1| 0.6371191143989563|
|       germany|2006|    1|  0.517082154750824|
|         india|2006|    1|0.28316405415534973|
|         italy|2006|    1| 0.1138811931014061|
|         japan|2006|    1|  0.129270538687706|
|        mexico|2006|    1|0.05847953259944916|
|   south_korea|2006|    1|0.23391813039779663|
|        russia|2006|    1| 0.6709756851196289|
|  south_africa|2006|    1|0.05232379212975502|
|       turkiye|2006|    1|0.05847953259944916|
|united_kingdom|2006|    1| 1.2065250873565674|
| united_states|2006|    1|   2.19144344329834|
|     australia|2006|    2|0.05412806198000908|
|        brazil|2006|    2|0.02865603193